# Analysis for the Prisoner's Dilemma with Punishment

In [76]:
import numpy as np
import pandas as pd

In [116]:
w = 0.9
mu = 0.1
N = 500

## Custom functions

In [117]:
# Computes fitness for A and B in a population of i A players and N-i B players. a and b refers to 
# the payoff strategy A earns against itself and against B respectively. c and d refers to
# the payoff B earns against A and B respectively

def f(a,b,w,N,i):
    return 1-w+w*((a*(i-1)+b*(N-i))/(N-1))

def g(c,d,w,N,i):
    return 1-w+w*((c*i+d*(N-i-1))/(N-1))

In [118]:
def fixation(a, b, c, d, w, N):
    den = 1
    for k in range(1, N):
        gamma = 1
        for i in range(1, k + 1):
            gamma *= g(c, d, w, N, i) / f(a, b, w, N, i)
        den += gamma
    return 1 / den

In [119]:
# Strategies considered
strategies = ["DNN", "CNN", "CPN", "FNN", "FPN"]
n = len(strategies)

# Payoff matrix for the five strategies (order above)
payoff_matrix = np.array([
    [1, 3, 1, 1, -1],   
    [0, 2, 2, 0, -2],   
    [-1, 2, 2, 2, 2],   
    [1, 3, 2, 1, 0],    
    [0, 3, 2, 3, 2],
])


# Matrix with pairwise fixation probability
fixation_matrix = np.zeros((len(strategies), len(strategies)))
for i in range(len(strategies)):
    for j in range(len(strategies)):
        if i != j:
            a = payoff_matrix[i][i]
            b = payoff_matrix[i][j]
            c = payoff_matrix[j][i]
            d = payoff_matrix[j][j]
            fixation_matrix[i][j] = fixation(a, b, c, d, w, N)


fixation_df = pd.DataFrame(fixation_matrix, index=strategies, columns=strategies)
print("Fixation Probability Matrix:")
print(fixation_df)

/var/folders/ys/ptz87krn7772h4x6tswpfs3w0000gn/T/ipykernel_15223/2059697401.py:7: RuntimeWarning: overflow encountered in scalar add
  den += gamma
/var/folders/ys/ptz87krn7772h4x6tswpfs3w0000gn/T/ipykernel_15223/2059697401.py:6: RuntimeWarning: overflow encountered in scalar multiply
  gamma *= g(c, d, w, N, i) / f(a, b, w, N, i)


Fixation Probability Matrix:
               DNN       CNN           CPN            FNN            FPN
DNN   0.000000e+00  0.323588  9.133841e-28   2.000000e-03  4.245874e-249
CNN  1.258409e-174  0.000000  2.000000e-03  1.258409e-174  7.007511e-274
CPN  3.705023e-161  0.002000  0.000000e+00   4.716670e-01   2.000000e-03
FNN   2.000000e-03  0.323588  4.502154e-63   0.000000e+00   0.000000e+00
FPN   9.264949e-54  0.328317  2.000000e-03   6.437308e-01   0.000000e+00


In [120]:
fixation_df.to_latex("fixation_matrix_2.tex",float_format="%.6f")

/var/folders/ys/ptz87krn7772h4x6tswpfs3w0000gn/T/ipykernel_15223/1182445057.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  fixation_df.to_latex("fixation_matrix_2.tex",float_format="%.6f")


In [121]:
# Transition matrix
T = pd.DataFrame(0.0, index=strategies, columns=strategies)

# Off-diagonal entries
for j in strategies:         
    for i in strategies:     
        if i != j:
            T.loc[j, i] = (mu/(n - 1)) * fixation_df.loc[i, j]

# Diagonal entries
for j in strategies:
    T.loc[j, j] = 1 - T.loc[j].drop(j).sum()

# Display or return T>
print("Transition matrix under rare mutations:")
print(T)

Transition matrix under rare mutations:
               DNN            CNN            CPN           FNN           FPN
DNN   9.999500e-01  3.146022e-176  9.262559e-163  5.000000e-05  2.316237e-55
CNN   8.089707e-03   9.755626e-01   5.000000e-05  8.089707e-03  8.207937e-03
CPN   2.283460e-29   5.000000e-05   9.999000e-01  1.125538e-64  5.000000e-05
FNN   5.000000e-05  3.146022e-176   1.179168e-02  9.720651e-01  1.609327e-02
FPN  1.061468e-250  1.751878e-275   5.000000e-05  0.000000e+00  9.999500e-01


In [122]:
T

,DNN,CNN,CPN,FNN,FPN
DNN,9.999500e-01,3.146022e-176,9.262559e-163,5.000000e-05,2.316237e-55
CNN,8.089707e-03,9.755626e-01,5.000000e-05,8.089707e-03,8.207937e-03
CPN,2.283460e-29,5.000000e-05,9.999000e-01,1.125538e-64,5.000000e-05
FNN,5.000000e-05,3.146022e-176,1.179168e-02,9.720651e-01,1.609327e-02
FPN,1.061468e-250,1.751878e-275,5.000000e-05,0.000000e+00,9.999500e-01


In [123]:
T.to_latex("transition_matrix_2.tex", float_format="%.6f")

/var/folders/ys/ptz87krn7772h4x6tswpfs3w0000gn/T/ipykernel_15223/1627241050.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  T.to_latex("transition_matrix_2.tex", float_format="%.6f")


## Stationary distribution

In [124]:
# Compute eigenvalues and eigenvectors
S, U = np.linalg.eig(T.T)

# Extract stationary distribution
stationary = (U[:, np.isclose(S, 1)][:, 0] / U[:, np.isclose(S, 1)][:, 0].sum()).real

# Convert to DataFrame
stationary_df = pd.DataFrame(stationary, index=strategies, columns=["Stationary Dist."])

print(stationary_df)

     Stationary Dist.
DNN          0.108804
CNN          0.000670
CPN          0.327499
FNN          0.000389
FPN          0.562638


In [125]:
# Save to LaTeX
with open("stationary_distribution_2.tex", "w") as f:
    f.write(stationary_df.to_latex(float_format="%.6f"))

/var/folders/ys/ptz87krn7772h4x6tswpfs3w0000gn/T/ipykernel_15223/2584980458.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  f.write(stationary_df.to_latex(float_format="%.6f"))


## Travelling time

In [ ]:
# Initial state: all DNN (index 0)
state = np.zeros(5)
state[0] = 1

# Track time steps until we're near certainty of being in FPN
threshold = 0.999  # You can adjust this as needed
max_steps = 1000
for t in range(1, max_steps + 1):
    state = state @ T
    if state[4] >= threshold:  # FPN is index 4
        print(f"Reached FPN with probability ≥ {threshold} at time step {t}")
        break
else:
    print("Did not reach the threshold within the max allowed steps.")
